# Meta-Model Prediction — Triple-Barrier Meta-Labeling + Model Search

Given a finished feature-engineering layer (`results/feature_matrix.parquet`), this notebook:

1. **Labels** each primary signal `1` (profitable) / `0` (not) with the **triple-barrier method**
   (meta-labeling: side = primary signal, volatility-scaled asymmetric barriers, vertical time-out).
2. **Tunes the barrier geometry** `(pt, sl, h)` on train+val only, by purged-CV validation AUC of a
   fixed baseline subject to a class-balance floor, preferring robust plateaus.
3. **Fits & compares** XGBoost, RandomForest, a PyTorch MLP and a PyTorch VSN-style net under
   **purged + embargoed walk-forward CV** with **Optuna**, across three scopes
   (pooled / per-asset-class / per-instrument).
4. **Analyses feature importance** for both families (SHAP / gain / permutation for trees;
   permutation / gradient saliency / selection-gates for the nets).
5. **Evaluates once on the held-out test partition.**

All logic lives in `src/stml/model/` (imported below); this notebook orchestrates and visualises.
The official test block is opened exactly once, via the `release_test` tripwire.

In [ ]:
%matplotlib inline
import warnings; warnings.filterwarnings("ignore")
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna

from stml.model import dataset as ds
from stml.model.labels import triple_barrier_labels, sample_uniqueness, class_balance
from stml.model.cv import PurgedWalkForward
from stml.model.barrier_search import search_barriers
from stml.model.optuna_objective import make_objective, run_study, cross_val_auc, MODEL_REGISTRY
from stml.model.trees import XGBModel, xgb_baseline_params
from stml.model.importance import tree_importance, nn_importance
from stml.model.evaluate import (evaluate_predictions, per_instrument_breakdown,
                                 plot_roc_pr, plot_calibration, release_test)

# ---- reproducibility + compute budget (raise the trial counts for a heavier run) ----
SEED = 0
np.random.seed(SEED)
N_TRIALS = {"xgb": 12, "rf": 10, "mlp": 6, "vsn": 4}
VAL_END = "2021-12-30"            # tuning never consults a price after this date
BARRIER_GRID = [(pt, sl, h)
                for h in (5, 10)
                for pt in (1.0, 1.5, 2.0)
                for sl in (1.0, 1.5, 2.0)]
MIN_MINORITY = 0.30
print(f"{len(BARRIER_GRID)} barrier configs | trials {N_TRIALS}")

## 1. Setup & provenance check
Load the frozen matrix and the clean close panel, attach each row's trading-bar position, and
confirm the FE-train boundary and split sizes match the provenance contract.

In [ ]:
matrix = ds.load_matrix()
close_wide = ds.close_panel()
matrix = ds.attach_bar_pos(matrix, close_wide)
scope_reg = ds.load_scope()
emb = ds.embargo_map()
FEATURE_COLS = ds.select_features(matrix)

prov = json.load(open(ds._results_dir() / "feature_matrix_provenance.json"))
assert prov["fe_train_end_date"] == "2021-07-01"
assert (matrix["bar_pos"] >= 0).all(), "every row must map onto its instrument's calendar"
print("FE-train boundary:", prov["fe_train_end_date"])
print("partition rows:", matrix["partition"].value_counts().to_dict())
print("feature cols (redundant dropped):", len(FEATURE_COLS), "| instruments:", matrix.instrument.nunique())
print("per-instrument embargo_p90:", emb)

## 2. Label interface
`side = f5_signal` (the primary signal) and `sigma = f2_vol_20` set the barriers. `f2_vol_20` is an
*annualised* vol, so `events_frame` de-annualises it to a per-bar return target. Below: the label
mix for a few barrier settings, and one worked example path.

In [ ]:
events = ds.events_frame(matrix)
print("sigma (per-bar) describe:", events["sigma"].describe()[["min","50%","max"]].round(4).to_dict())
for pt, sl, h in [(1.0,1.0,5),(1.5,1.5,10),(2.0,1.0,10)]:
    lab = triple_barrier_labels(close_wide, events, pt=pt, sl=sl, h=h, price_end=VAL_END)
    b = class_balance(lab); t = lab.touch.value_counts(normalize=True).round(2).to_dict()
    print(f"pt={pt} sl={sl} h={h}: n={b['n']} pos_rate={b['pos_rate']:.2f} "
          f"minority={b['minority_frac']:.2f} touch={t}")

In [ ]:
# worked example: one long-side event on es1s, show entry + PT/SL bands over its window
ex = matrix[(matrix.instrument=="es1s") & (matrix.f5_signal==1)].iloc[200]
s = close_wide["es1s"].dropna(); p = s.index.get_loc(ex["date"])
sig = ex["f2_vol_20"]/np.sqrt(252); H=10
seg = s.iloc[p:p+H+1]; entry = seg.iloc[0]
fig, ax = plt.subplots(figsize=(8,4))
ax.plot(seg.index, seg.values, "o-", label="close")
ax.axhline(entry*(1+1.5*sig), ls="--", c="g", label="+1.5σ PT")
ax.axhline(entry*(1-1.5*sig), ls="--", c="r", label="-1.5σ SL")
ax.axhline(entry, c="k", lw=0.8); ax.set_title(f"es1s long event {ex['date'].date()} — triple barrier")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

## 3. Purged walk-forward CV scaffold
Development = train+val partitions only. Folds are expanding; per-instrument purge (`h`) + embargo
(`embargo_p90`) open the train-side gap before each validation block. The test partition is never
passed here.

In [ ]:
dev_matrix = matrix[matrix.partition.isin(ds.DEV_PARTITIONS)].reset_index(drop=True)
cv_demo = PurgedWalkForward(n_splits=4, h=10, embargo_by_instrument=emb)
folds = list(cv_demo.split(dev_matrix))
fig, ax = plt.subplots(figsize=(9,3))
for i,(tr,va) in enumerate(folds):
    ax.scatter(dev_matrix.iloc[tr]["date"], np.full(tr.size, i), s=3, c="tab:blue")
    ax.scatter(dev_matrix.iloc[va]["date"], np.full(va.size, i), s=3, c="tab:orange")
ax.set_yticks(range(4)); ax.set_ylabel("fold"); ax.set_title("Expanding folds (blue=train, orange=val)")
plt.tight_layout(); plt.show()
print(pd.DataFrame(cv_demo.fold_n_eff(dev_matrix.assign(side=dev_matrix[ds.SIDE_COL]))))

# confirm the test tripwire is armed
try:
    release_test(matrix); print("TRIPWIRE FAILED")
except RuntimeError:
    print("test tripwire armed — release_test refuses access without confirmation")

## 4. Barrier-geometry search
Score each `(pt, sl, h)` by mean purged-CV AUC of a fixed shallow XGBoost on the pooled panel
(subject to the minority ≥ 30% floor), then pick the **robust plateau** — the config whose
neighbourhood, not just itself, scores well.

In [ ]:
br = search_barriers(dev_matrix, close_wide, grid=BARRIER_GRID, price_end=VAL_END,
                     min_minority=MIN_MINORITY, seed=SEED, feature_cols=FEATURE_COLS)
disp = br.table[["pt","sl","h","minority","auc","auc_std","plateau","skipped"]].round(3)
print(disp.to_string(index=False))
PT, SL, H = float(br.best["pt"]), float(br.best["sl"]), int(br.best["h"])
print(f"\nCHOSEN barriers: pt={PT} sl={SL} h={H} | plateau AUC={br.best['plateau']:.3f} "
      f"(self AUC={br.best['auc']:.3f}) | configs tried={br.n_configs} (deflate accordingly)")

In [ ]:
# plateau-AUC heatmaps over (pt, sl) at each h
hs = sorted(br.table.h.unique())
fig, axes = plt.subplots(1, len(hs), figsize=(5*len(hs),4), squeeze=False)
for ax, h in zip(axes[0], hs):
    piv = br.table[br.table.h==h].pivot(index="sl", columns="pt", values="plateau")
    im = ax.imshow(piv.values, origin="lower", aspect="auto", cmap="viridis")
    ax.set_xticks(range(len(piv.columns))); ax.set_xticklabels(piv.columns)
    ax.set_yticks(range(len(piv.index))); ax.set_yticklabels(piv.index)
    ax.set_xlabel("pt"); ax.set_ylabel("sl"); ax.set_title(f"plateau AUC, h={h}")
    fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()

## 5. Dataset assembly (chosen barriers)
Relabel the development panel with the frozen barriers, attach López de Prado sample-uniqueness
weights, and inspect class balance per scope.

In [ ]:
dev_labels = triple_barrier_labels(close_wide, ds.events_frame(dev_matrix),
                                   pt=PT, sl=SL, h=H, price_end=VAL_END)
w = sample_uniqueness(dev_labels, close_wide)
dev_lab = dev_matrix.merge(
    dev_labels[["date","instrument","bin"]].assign(w=w.values),
    on=["date","instrument"], how="inner").reset_index(drop=True)
print("dev labeled rows:", len(dev_lab), "| overall minority:", round(class_balance(dev_lab)["minority_frac"],3))

ac = ds.asset_class_map(scope_reg)
bal = (dev_lab.assign(asset_class=dev_lab.instrument.map(ac))
       .groupby(["asset_class","instrument"])
       .agg(n=("bin","size"), pos_rate=("bin","mean")).round(3))
print(bal)
# per-instrument viability for the per-instrument scope
counts = dev_lab.instrument.value_counts()
MIN_CELL = 120
SKIP_INSTR = sorted([i for i in counts.index
                     if counts[i] < MIN_CELL or scope_reg[i]["low_power"]])
print("per-instrument cells skipped (thin / low-power, folded into class):", SKIP_INSTR)

## 6. Model tuning — {XGB, RF, MLP, VSN} × {pooled, per-class, per-instrument}
Each cell runs an Optuna study maximising mean purged-CV AUC on the shared folds. Trees get
uniqueness `sample_weight`; NNs are restricted to the pooled and per-class scopes (per-instrument
samples are too thin), and the thin per-instrument cells are skipped for trees too.

In [ ]:
def tune_cell(scope, model_key, cell_id, cell):
    cell = cell.reset_index(drop=True)
    X, y = ds.make_xy(cell, FEATURE_COLS, instrument_dummies=(scope != "per_instrument"))
    if np.unique(y).size < 2:
        return None
    cv = PurgedWalkForward(n_splits=4, h=H, embargo_by_instrument=emb)
    sw = cell["w"].to_numpy() if model_key in ("xgb", "rf") else None
    study = run_study(make_objective(model_key, X, y, cell, cv, seed=SEED, sample_weight=sw),
                      N_TRIALS[model_key], seed=SEED)
    return {"scope": scope, "model": model_key, "cell": cell_id, "n": int(len(y)),
            "cv_auc": float(study.best_value),
            "cv_std": float(study.best_trial.user_attrs.get("auc_std", float("nan"))),
            "n_folds": int(study.best_trial.user_attrs.get("n_folds", 0)),
            "params": study.best_params}

results = []
for scope in ["pooled", "per_class", "per_instrument"]:
    models = ["xgb", "rf"] if scope == "per_instrument" else ["xgb", "rf", "mlp", "vsn"]
    for cell_id, cell in ds.scope_iter(dev_lab, scope, scope_reg=scope_reg):
        if scope == "per_instrument" and cell_id in SKIP_INSTR:
            continue
        for mk in models:
            try:
                r = tune_cell(scope, mk, cell_id, cell)
                if r:
                    results.append(r)
                    print(f"{scope:15s} {mk:4s} {cell_id:8s} n={r['n']:4d} "
                          f"AUC={r['cv_auc']:.3f}±{r['cv_std']:.3f} folds={r['n_folds']}")
            except Exception as e:
                print(f"{scope} {mk} {cell_id} FAILED: {e}")
results = pd.DataFrame(results)

## 7. Cross-scope comparison
Aggregate per-cell CV AUC to a scope × model view (sample-size-weighted across cells), and pick the
overall winner. A robust choice has a high mean **and** a low fold-to-fold std.

In [ ]:
summ_rows = []
for (scope, model), g in results.groupby(["scope","model"]):
    summ_rows.append({"scope": scope, "model": model,
                      "wavg_auc": float(np.average(g["cv_auc"], weights=g["n"])),
                      "mean_std": float(g["cv_std"].mean()), "cells": int(len(g))})
summary = pd.DataFrame(summ_rows)
piv = summary.pivot(index="model", columns="scope", values="wavg_auc").round(3)
print("Sample-weighted mean CV AUC (model × scope):"); print(piv)

fig, ax = plt.subplots(figsize=(8,4)); piv.plot.bar(ax=ax)
ax.axhline(0.5, c="k", ls="--", lw=0.8); ax.set_ylabel("weighted CV AUC")
ax.set_ylim(0.45, max(0.62, piv.max().max()+0.02)); ax.set_title("Meta-model AUC by scope")
ax.legend(title="scope", fontsize=8); plt.tight_layout(); plt.show()

# overall winner: prefer pooled (one model, all instruments), robust by AUC minus 0.5*std
cand = results.assign(score=results.cv_auc - 0.5*results.cv_std.fillna(0))
best_overall = cand.sort_values("score", ascending=False).iloc[0]
print(f"\nBest single cell: {best_overall['model']} / {best_overall['scope']} / "
      f"{best_overall['cell']} — CV AUC {best_overall['cv_auc']:.3f}±{best_overall['cv_std']:.3f}")

## 8. Feature importance (trees & neural nets)
Refit the best **pooled** model of each family on the full development panel and inspect importance.
We expect the counter-trend F1 family (`f1_mr_*`, `f1_ret_reversal_*`) to rank high — the released
signal is counter-trend per the feature catalog.

In [ ]:
pooled = dev_lab.reset_index(drop=True)
Xp, yp = ds.make_xy(pooled, FEATURE_COLS, instrument_dummies=True)
swp = pooled["w"].to_numpy()

def refit_best(model_key, scope="pooled"):
    sub = results[(results.model==model_key) & (results.scope==scope)]
    if sub.empty: return None, None
    best = sub.sort_values("cv_auc", ascending=False).iloc[0]
    cls, space = MODEL_REGISTRY[model_key]
    params = space(optuna.trial.FixedTrial(best["params"]))
    sw = swp if model_key in ("xgb","rf") else None
    model = cls(params, SEED).fit(Xp, yp, sample_weight=sw)
    return model, best["cv_auc"]

def top_plot(imp, title, col):
    top = imp.sort_values(col, ascending=False).head(20)
    fig, ax = plt.subplots(figsize=(7,6))
    ax.barh(top.index[::-1], top[col].values[::-1]); ax.set_title(title); ax.tick_params(labelsize=8)
    plt.tight_layout(); plt.show()

# best tree family
tree_key = "xgb" if (results[(results.scope=="pooled")&(results.model=="xgb")]["cv_auc"].max()
                     >= results[(results.scope=="pooled")&(results.model=="rf")]["cv_auc"].max()) else "rf"
tmodel, tauc = refit_best(tree_key)
samp = Xp.sample(min(1500, len(Xp)), random_state=SEED)
samp_y = yp[samp.index.to_numpy()]
timp = tree_importance(tmodel, samp, samp_y, n_repeats=3, seed=SEED)
print(f"Top tree features ({tree_key}, pooled CV AUC {tauc:.3f}):"); print(timp.head(15).round(4))
top_plot(timp, f"{tree_key} — mean |SHAP|", "shap")

In [ ]:
# best neural family (mlp vs vsn, pooled)
nn_key = "vsn" if (results[(results.scope=="pooled")&(results.model=="vsn")]["cv_auc"].max()
                   >= results[(results.scope=="pooled")&(results.model=="mlp")]["cv_auc"].max()) else "mlp"
nmodel, nauc = refit_best(nn_key)
nimp = nn_importance(nmodel, samp, samp_y, n_repeats=3, seed=SEED)
print(f"Top NN features ({nn_key}, pooled CV AUC {nauc:.3f}):"); print(nimp.head(15).round(4))
top_plot(nimp, f"{nn_key} — permutation AUC drop", "perm_auc_drop")

# do the counter-trend F1 features surface near the top?
f1_rank = [f for f in timp.head(20).index if f.startswith("f1_")]
print("F1 (counter-trend) features in tree top-20:", f1_rank)

## 9. Final evaluation on the held-out test partition
Opened once, via the tripwire. Test events are labelled with the full price history (their own
realised future); the chosen-barrier dev models are refit on all of train+val and scored on test.

In [ ]:
test_matrix = release_test(matrix, final_confirmation=True)
test_labels = triple_barrier_labels(close_wide, ds.events_frame(test_matrix), pt=PT, sl=SL, h=H)
test_lab = test_matrix.merge(test_labels[["date","instrument","bin"]],
                             on=["date","instrument"], how="inner").reset_index(drop=True)
Xte, yte = ds.make_xy(test_lab, FEATURE_COLS, instrument_dummies=True)
Xte = ds.align_columns(Xte, list(Xp.columns))
print("test labeled rows:", len(test_lab), "| pos_rate:", round(yte.mean(),3))

rows, curves = [], {}
for mk in ["xgb","rf","mlp","vsn"]:
    model, cvauc = refit_best(mk)
    if model is None: continue
    proba = model.predict_proba(Xte)
    m = evaluate_predictions(yte, proba); m.update(model=mk, pooled_cv_auc=round(cvauc,3))
    rows.append(m); curves[mk] = (yte, proba)
test_tbl = pd.DataFrame(rows).set_index("model")[
    ["pooled_cv_auc","auc","ap","f1","precision","recall","brier","n"]].round(3)
print("TEST metrics (pooled models):"); print(test_tbl)

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(12,5))
plot_roc_pr(curves, ax=axes[0])
winner = test_tbl["auc"].idxmax()
wmodel, _ = refit_best(winner)
plot_calibration(yte, wmodel.predict_proba(Xte), ax=axes[1])
axes[1].set_title(f"Calibration — {winner}")
plt.tight_layout(); plt.show()

bd = per_instrument_breakdown(test_lab["instrument"].to_numpy(), yte, wmodel.predict_proba(Xte))
print(f"Per-instrument TEST AUC — winner = {winner}:"); print(bd.round(3).to_string(index=False))

## 10. Conclusions & limitations

- **Barriers.** Frozen on train+val by robust-plateau purged-CV AUC; the deflation note (configs
  tried) should temper any single-config enthusiasm.
- **Models & scopes.** The scope × model table shows where signal lives; pooling gives the most
  data per model and the per-instrument breakdown reveals which names carry it.
- **Feature importance.** Cross-checked against the catalog's counter-trend (F1) expectation.
- **Limitations.** (1) ~2.5-year signal window → modest, fragile AUC; treat test as one draw.
  (2) Overlapping labels are only partly addressed by uniqueness weights (no full CPCV).
  (3) Thin instruments (ho1s, ng1s, gc1s) are folded into class scope — their standalone test AUC
  is uninterpretable. (4) The VSN uses a shared feature-GRN for speed (a common simplification).